# Synthea Explorations

This notebook explores synthetic patient data generated by [Synthea](https://github.com/synthetichealth/synthea).
Synthea outputs FHIR R4 Bundles containing Patient, Practitioner, Encounter, Condition, and other resources.

In [1]:
import os

from dicom_py_mock_server.services.synthea_utils import SyntheaUtils

SYNTHEA_OUTPUT_FOLDER = "../../../synthea/output/fhir"
MAX_BUNDLES = 75


# Check if the output folder exists
if not os.path.exists(SYNTHEA_OUTPUT_FOLDER):
    print(f"Output folder '{SYNTHEA_OUTPUT_FOLDER}' does not exist.")
else:
    print(f"Output folder '{SYNTHEA_OUTPUT_FOLDER}' found.")

# Load the Synthea output data, parse them into a list of FHIR Bundle dicts, and return the list

syn = SyntheaUtils(synthea_output_folder=SYNTHEA_OUTPUT_FOLDER)

Output folder '../../../synthea/output/fhir' found.


In [12]:
from ast import List

from dicom_py_mock_server.models.fhir_models import (
    FhirOrganization,
    FhirPractitioner,
    format_human_name_to_dicom,
)

hospitals: List[FhirOrganization] = syn.get_hospital_information()
org: FhirOrganization
practitioners: List[FhirPractitioner]
p: FhirPractitioner
practitioners = syn.get_practioner_information()
if not hospitals:
    print("Failed to get hospitals")
if not practitioners:
    print("Failed to get practitioners")
for org in hospitals:
    # print(f"{org.id}: {org.name} <- {org.partOf}")
    pass
for p in practitioners:
    name = format_human_name_to_dicom(p.name[0])
    ident = p.identifier[0].value
    print(f"{name} NPI: {ident}")

Reilly981^Ted955^^Dr. NPI: 9999984591
Ward668^Olympia319^^Dr. NPI: 9999928192
Emard19^Jules135^^Dr. NPI: 9999990390
Lehner980^Gisele901^^Dr. NPI: 9999984997
Armijo730^Bernardo699^^Dr. NPI: 9999995696
Leannon79^Thad495^^Dr. NPI: 9999889493
Ankunding277^Avis183^^Dr. NPI: 9999949792
Hettinger594^Arlen68^^Dr. NPI: 9999957993
Parisian75^Bud153^^Dr. NPI: 9999939991
McCullough561^Archie818^^Dr. NPI: 9999894790
Toy286^Anthony633^^Dr. NPI: 9999978395
Champlin946^Tracey100^^Dr. NPI: 9999945097
Fahey393^Maxima831^^Dr. NPI: 9999900993
Howe413^Kurt412^^Dr. NPI: 9999874297
Terry864^Clifton91^^Dr. NPI: 9999969998
Hudson301^Jennefer743^^Dr. NPI: 9999899591
Little434^Ronny764^^Dr. NPI: 9999911495
Schuster709^Dana512^^Dr. NPI: 9999976696
Mosciski958^Gussie514^^Dr. NPI: 9999972497
Auer97^Hallie315^^Dr. NPI: 9999891093
Corwin846^Carylon722^^Dr. NPI: 9999898197
Caballero789^Maricarmen445^^Dr. NPI: 9999938399
Kreiger457^Rodrick370^^Dr. NPI: 9999873190
Tromp100^Delbert384^^Dr. NPI: 9999898999
Ziemann98^Azzie

In [ ]:
from ast import List

from fhir.resources.R4B.diagnosticreport import DiagnosticReport
from fhir.resources.R4B.patient import Patient
from fhir.resources.R4B.procedure import Procedure

bundles = syn.scan_synthea_output(modalities=["CT", "US", "CR"], max_count=MAX_BUNDLES)

print(f"Loaded {len(bundles)} FHIR Bundles from Synthea output.")

for bundle in bundles:
    patient: Patient = syn.get_patient(bundle)
    dicom_name = syn.get_patient_name(patient)
    patient_id = syn.get_patient_id(patient)
    sex = syn.get_patient_sex(patient)
    birth_date = syn.get_date_time(patient.birthDate)
    print("\n====================================")
    print(f"Patient: DICOM Name: {dicom_name}, Gender: {sex}, ID: {patient_id}, Birth Date: {birth_date[0]}")
    # Imaging Studies
    imagings = syn.get_imaging_studies(bundle)
    for imaging in imagings:
        for s in imaging.series:
            modality = s.modality.display
            print("-- Imaging --")
            print(f"Modality: {modality}")
        enc_ref = imaging.encounter.reference
        encounters = syn.get_encounters(bundle=bundle, encounter_ref=enc_ref)
        for enc in encounters:
            print(f"Encounter On:{enc.period.start}: {enc_ref} {enc.id}")
        diags: List[DiagnosticReport] = syn.get_diagnostic_reports(bundle=bundle, encounter_ref=enc_ref)
        procedures: List[Procedure] = syn.get_procedures(bundle=bundle, encounter_ref=enc_ref)
        diag: DiagnosticReport
        proc: Procedure
        for diag in diags:
            if diag and diag.presentedForm:
                for form in diag.presentedForm:
                    match form:
                        case Attachment as a:
                            print(f"Diagnostic Report: {a.data.decode('utf-8')}")
        for proc in procedures:
            code = proc.code.text
            reason = ""
            if proc.reasonReference:
                reason = proc.reasonReference[0].display
            location = proc.location.display
            print(f"Procedure: {code}, {reason} at {location}")

Loaded 8 FHIR Bundles from Synthea output.

Patient: DICOM Name: Hoppe518^Graham902^Robby860^Mr., Gender: M, ID: S99986230, Birth Date: 19180605
-- Imaging --
Modality: Ultrasound
Encounter On:1995-05-17 00:37:39-07:00: urn:uuid:0498f608-29de-1be2-aef7-ed701b7df3ae 0498f608-29de-1be2-aef7-ed701b7df3ae
Diagnostic Report: 
1995-05-17

# Chief Complaint
- Thirst
- Confusion
- Fatigue
- Decreased Communication
- Decreased Attentiveness
- urinary frequency


# History of Present Illness
Graham902 is a 76 year-old nonhispanic asian male. Patient has a history of part-time employment (finding), medication review due (situation), not in labor force (finding), full-time employment (finding), victim of intimate partner abuse (finding), gingivitis (disorder), stress (finding), social isolation (finding).

# Social History
Patient is married. Patient has never smoked.
 Patient identifies as heterosexual.

Patient comes from a middle socioeconomic background.
 Patient is a college graduate.
Patient